# Zero-Shot Linking Demo

This notebook loads a released IdentityPFN checkpoint, scores a small table of toy records, and inspects the predicted adjacency matrix.

In [ ]:
from pathlib import Path
import os

if not Path("identitypfn").exists() and Path("../identitypfn").exists():
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from identitypfn import load_model, nearest_neighbors, pair_review_table, top_pairs

## Toy Records

In [ ]:
records = pd.DataFrame(
    [
        {"first_name": "Ada", "last_name": "Lovelace", "email": "ada@example.com", "phone": "415-555-0101", "city": "San Francisco"},
        {"first_name": "A.", "last_name": "Lovelace", "email": "ada@example.com", "phone": "+1 415 555 0101", "city": "SF"},
        {"first_name": "Grace", "last_name": "Hopper", "email": "grace@example.com", "phone": "212-555-0199", "city": "New York"},
        {"first_name": "Grace M", "last_name": "Hopper", "email": "g.hopper@example.com", "phone": "2125550199", "city": "NYC"},
        {"first_name": "Katherine", "last_name": "Johnson", "email": "kj@example.com", "phone": "703-555-0188", "city": "Hampton"},
        {"first_name": "Alan", "last_name": "Turing", "email": "alan@example.org", "phone": "020 5555 0177", "city": "London"},
    ],
    index=["r0", "r1", "r2", "r3", "r4", "r5"],
)
field_types = ["text", "text", "email", "phone", "text"]
records

## Load Checkpoint and Score All Pairs

In [ ]:
checkpoint_path = Path("results/model_checkpoints/20260715_195803_seed1337_20005214_best_step1500.pt")
linker = load_model(checkpoint_path, device="cpu")
scores = linker.predict_proba(records, field_types)
scores.round(3)

## Adjacency Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(scores, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(scores.columns)), scores.columns)
ax.set_yticks(range(len(scores.index)), scores.index)
ax.set_title("Predicted pairwise match probabilities")
fig.colorbar(image, ax=ax, label="match probability")
fig.tight_layout()

## Top Pair Search

In [ ]:
top_pairs(scores, k=8)

## Per-Record Nearest Neighbors

In [ ]:
nearest_neighbors(scores, k=2)

## Review Table

In [ ]:
pair_review_table(
    scores,
    records,
    k=8,
    columns=["first_name", "last_name", "email", "phone", "city"],
).round({"score": 3})